# 1. Dataclasses

Dataclasses are a compact way to create classes whose main purpose is to store data. They reduce boilerplate by generating common methods automatically, while still letting you add validation, defaults, ordering, and immutability when needed.

## Prerequisites

You should already know:

- basic Python syntax
- functions and classes
- basic types such as `int`, `float`, `str`, `list`, and `dict`

Run the cells step by step and compare the dataclass examples with the regular class examples.

## 1.1 What Is a Dataclass?

A dataclass is a class designed mainly for holding data. The `dataclasses` module can generate methods such as `__init__`, `__repr__`, and `__eq__` for you, and it can optionally generate ordering methods as well.

Dataclasses are most useful when:

- the class mostly stores related values
- you want readable representations for debugging
- you want value-based equality without writing extra boilerplate

They are less useful when the class is mostly complex behavior with very little stored state.

In [1]:
from dataclasses import dataclass

@dataclass
class Student:
    name: str
    age: int
    grade: float

s = Student("Alice", 20, 15.5)
s

Student(name='Alice', age=20, grade=15.5)

Observe that we did **not** write `__init__` or `__repr__`. The `@dataclass` decorator generated them automatically.

## 2. Basic Syntax

### 2.1 Defining a Dataclass

Key points:

- apply the `@dataclass` decorator above the class
- declare fields with type hints
- create instances the same way you create instances of a regular class

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

p = Point(1.5, 3.2)
p

### 2.2 Default Values

You can give default values to fields, just like normal function parameters.


In [ ]:
from dataclasses import dataclass

@dataclass
class StudentWithDefaults:
    name: str
    age: int = 18
    passed: bool = False

s1 = StudentWithDefaults("Bob")
s2 = StudentWithDefaults("Clara", age=21, passed=True)
s1, s2

## 3. Dataclass Versus Regular Class

A regular class can do the same job, but it usually needs more setup code. Dataclasses are helpful when that extra code is mostly repetitive boilerplate rather than custom behavior.

In [ ]:
class StudentClassic:
    def __init__(self, name: str, age: int, grade: float):
        self.name = name
        self.age = age
        self.grade = grade

    def __repr__(self):
        return f"StudentClassic(name={self.name!r}, age={self.age}, grade={self.grade})"

classic = StudentClassic("Alice", 20, 16.0)
classic

In [ ]:
from dataclasses import dataclass

@dataclass
class StudentData:
    name: str
    age: int
    grade: float

data = StudentData("Alice", 20, 16.0)
data

## 4. Generated Methods

Dataclasses automatically generate several methods for you.

Common ones include:

- `__init__` for object creation
- `__repr__` for readable object display
- `__eq__` for value-based comparison

### 4.1 Equality Example

Unlike a plain custom class, two dataclass instances with the same field values compare as equal by default.

In [ ]:
s1 = StudentData("Alice", 20, 16.0)
s2 = StudentData("Alice", 20, 16.0)

s1 == s2  # compares by fields, not by identity (address in memory)


## 5. `field()` for More Control

The `field()` function lets you customize how individual fields behave, including defaults, representation, and comparison.

### 5.1 Mutable Defaults with `default_factory`

Avoid mutable defaults such as `[]` directly in the class definition. Otherwise, instances may accidentally share the same object.

Use `default_factory` instead so each instance receives its own fresh value.

In [2]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Classroom:
    name: str
    students: List[str] = field(default_factory=list)

room1 = Classroom("Room A")
room2 = Classroom("Room B")

room1.students.append("Alice")
room2.students.append("Bob")

room1, room2  # different lists, as expected


(Classroom(name='Room A', students=['Alice']),
 Classroom(name='Room B', students=['Bob']))

### 5.2 Hiding a Field from `__repr__`

Use `repr=False` when a field is sensitive or not useful in textual output.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class User:
    username: str
    password: str = field(repr=False)

u = User("john", "secret123")
u  # password is hidden in the representation


### 5.3 Ignoring a Field in Comparisons

With `compare=False`, a field is ignored when checking equality.


In [ ]:
from dataclasses import dataclass, field
import time

@dataclass
class Measurement:
    value: float
    unit: str = "m"
    timestamp: float = field(compare=False, default_factory=time.time)

m1 = Measurement(10.0)
m2 = Measurement(10.0)

m1 == m2  # True (timestamp is ignored in comparison)


## 6. `__post_init__` for Derived Values and Validation

`__post_init__` runs immediately after the autogenerated `__init__`. It is the right place for work that depends on the initialized fields.

Common uses include:

- computing derived values
- validating inputs
- normalizing incoming data

In [3]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Product:
    name: str
    price: float
    discounted_price: Optional[float] = None

    def __post_init__(self):
        if self.discounted_price is None:
            self.discounted_price = self.price

p = Product("Book", 10.0)
p  # discounted_price is set automatically


Product(name='Book', price=10.0, discounted_price=10.0)

### 6.1 Validation Example

We can validate fields and raise errors when values are invalid.


In [ ]:
from dataclasses import dataclass

@dataclass
class StudentValidated:
    name: str
    age: int
    grade: float

    def __post_init__(self):
        if self.age < 0:
            raise ValueError("Age cannot be negative")
        if not (0 <= self.grade <= 20):
            raise ValueError("Grade must be between 0 and 20")

# Try a valid one
StudentValidated("Alice", 20, 15.0)

# Uncomment the next line to see the exception
# StudentValidated("Bob", -1, 10.0)


## 7. Ordering Dataclass Instances

Using `@dataclass(order=True)` tells Python to generate ordering methods such as `<`, `<=`, `>`, and `>=`. By default, instances are compared field by field in the order the fields are declared.

In [4]:
from dataclasses import dataclass

@dataclass(order=True)
class Player:
    score: int
    name: str

players = [
    Player(10, "Alice"),
    Player(5, "Bob"),
    Player(20, "Clara"),
]

sorted(players)  # sorted by score (then name if scores equal)


[Player(score=5, name='Bob'),
 Player(score=10, name='Alice'),
 Player(score=20, name='Clara')]

## 8. Immutability with `frozen=True`

With `frozen=True`, dataclass instances become read-only after creation. This is useful when the object should behave like a fixed value rather than a mutable container.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class PointFrozen:
    x: float
    y: float

pf = PointFrozen(1.0, 2.0)
pf

# Uncomment to see the error
# pf.x = 3.0  # dataclasses.FrozenInstanceError


## 9. Converting to Dictionaries and Tuples

Dataclasses provide helper functions such as `asdict()` and `astuple()`. These are useful when you want to serialize, inspect, print, or pass structured data to other code.

In [ ]:
from dataclasses import dataclass, asdict, astuple

@dataclass
class SimpleStudent:
    name: str
    age: int

ss = SimpleStudent("Alice", 20)
d = asdict(ss)
t = astuple(ss)

d, t


You can reconstruct from the dictionary using `**` unpacking:


In [ ]:
ss2 = SimpleStudent(**d)
ss2, ss2 == ss


## 10. Inheritance with Dataclasses

Dataclasses can inherit from other dataclasses. The main rule to remember is that non-default fields must still appear before default fields across the full inheritance chain.

In [ ]:
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int

@dataclass
class StudentInherited(Person):
    grade: float

si = StudentInherited("Alice", 20, 15.5)
si


## 11. Common Pitfalls

Watch for these common mistakes when using dataclasses:

1. Mutable default values: use `field(default_factory=...)` instead of a shared list or dict.
2. Missing type hints: dataclasses discover fields from annotations.
3. Wrong field order: non-default fields must come before default fields.
4. Forgetting immutability rules: `frozen=True` prevents later modification.

## 12. Practice Exercises

Use the cells below to write and run your own solutions.

### Exercise 1: Basic Dataclass

Create a dataclass `Book` with:

- `title: str`
- `author: str`
- `price: float`
- `in_stock: bool = True`

Then create three instances, store them in a list, and inspect the autogenerated representation.

In [ ]:
# TODO: Exercise 1 solution here

from dataclasses import dataclass

# Your code below

pass


## Exercise 2 – Classroom with Students

Create:
```python
@dataclass
class Student:
    name: str
    grade: float

@dataclass
class Classroom:
    name: str
    students: list[Student] = field(default_factory=list)
```

Tasks:
1. Add a method `add_student(self, student: Student)` to `Classroom`.
2. Add a method `average_grade(self) -> float` that returns the average grade (0 if no students).
3. Test with several students.


In [ ]:
# TODO: Exercise 2 solution here

from dataclasses import dataclass, field
from typing import List

# Your code below

pass


## Exercise 3 – Validation with `__post_init__`

Extend `Student` so that:
- Grade must be between 0 and 20.
- If not, raise `ValueError` in `__post_init__`.

Test by creating a `Student` with invalid grade and observe the error.


In [ ]:
# TODO: Exercise 3 solution here

# Your code below

pass


## Exercise 4 – Immutable Configuration

Create a frozen dataclass:
```python
@dataclass(frozen=True)
class Config:
    host: str
    port: int
```

Try to modify `host` after creation and observe the error.


In [ ]:
# TODO: Exercise 4 solution here

from dataclasses import dataclass

# Your code below

pass


## Exercise 5 – Sorting Players

Create:
```python
@dataclass(order=True)
class Player:
    score: int
    name: str
```

1. Create several players with different scores.
2. Sort them and print them.
3. (Optional) Add a field `level` that is **not** used for comparison (use `compare=False`).


In [ ]:
# TODO: Exercise 5 solution here

from dataclasses import dataclass, field

# Your code below

pass


## 13. Summary

In this notebook, you learned how to:

- define dataclasses with `@dataclass`
- use defaults and `field()`
- use `__post_init__` for validation and computed values
- enable ordering and immutability
- convert dataclasses to and from dictionaries and tuples
- avoid the most common mistakes

Dataclasses are a strong default choice when a class mainly stores data and you want clearer, shorter, and more maintainable code.